<a href="https://colab.research.google.com/github/songys/learning-langchain/blob/main/python/ch01/Learning_LangChain_Ch01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1장. 랭체인 기초

In [1]:
# ⏱ 패키지 설치 (1~2분 소요)
# langchain: LLM 애플리케이션 프레임워크
# langchain-ollama: Ollama 모델 연동 (ChatOllama, OllamaLLM)
# langchain-community: 서드파티 통합 도구
!pip install -q langchain langchain-ollama langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
# OpenAI API는 사용하지 않습니다 (Ollama 사용)
# from google.colab import userdata
# import os

# os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')

In [3]:
# ⏱ Ollama 설치 및 모델 다운로드 (3~5분 소요)
import subprocess
import time

!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

subprocess.Popen(['ollama', 'serve'])
time.sleep(3)

!ollama pull llama3.2

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (320 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current use

## 코드 1-1 기본 LLM 호출

In [4]:
from langchain_ollama import OllamaLLM

model = OllamaLLM(model="llama3.2")

model.invoke("The sky is")

'blue.'

## 코드 1-2 채팅 모델 호출

In [5]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

model = ChatOllama(model="llama3.2")
prompt = [HumanMessage("What is the capital of France?")]

model.invoke(prompt)


AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-03-08T05:26:21.911593715Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2945614178, 'load_duration': 165251684, 'prompt_eval_count': 32, 'prompt_eval_duration': 1322713874, 'eval_count': 8, 'eval_duration': 1452609430, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--019ccbe9-2054-7f51-a273-79c2d62651e4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 8, 'total_tokens': 40})

## 코드 1-3 시스템 메시지를 적용한 채팅 모델 호출

In [6]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

model = ChatOllama(model="llama3.2")
system_msg = SystemMessage(
    '''You are a helpful assistant that responds to questions with three
        exclamation marks.'''
)
human_msg = HumanMessage('What is the capital of France?')

model.invoke([system_msg, human_msg])

AIMessage(content='!!!Paris!!!', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-03-08T05:26:26.962946849Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4986351157, 'load_duration': 142820805, 'prompt_eval_count': 49, 'prompt_eval_duration': 4261111232, 'eval_count': 4, 'eval_duration': 579191002, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--019ccbe9-2c17-7a72-9cab-07b6dc19a4e2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 4, 'total_tokens': 53})

## 코드 1-4 프롬프트 템플릿 적용

In [7]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("""Answer the question based on the
    context below. If the question cannot be answered using the information
    provided, answer with "I don't know".

Context: {context}

Question: {question}

Answer: """)

template.invoke({
    "context": """The most recent advancements in NLP are being driven by Large
        Language Models (LLMs). These models outperform their smaller
        counterparts and have become invaluable for developers who are creating
        applications with NLP capabilities. Developers can tap into these
        models through Hugging Face's `transformers` library, or by utilizing
        OpenAI and Cohere's offerings through the `openai` and `cohere`
        libraries, respectively.""",
    "question": "Which model providers offer LLMs?"
})

StringPromptValue(text='Answer the question based on the\n    context below. If the question cannot be answered using the information\n    provided, answer with "I don\'t know".\n\nContext: The most recent advancements in NLP are being driven by Large\n        Language Models (LLMs). These models outperform their smaller\n        counterparts and have become invaluable for developers who are creating\n        applications with NLP capabilities. Developers can tap into these\n        models through Hugging Face\'s `transformers` library, or by utilizing\n        OpenAI and Cohere\'s offerings through the `openai` and `cohere`\n        libraries, respectively.\n\nQuestion: Which model providers offer LLMs?\n\nAnswer: ')

## 코드 1-5 동적 프롬프트

In [8]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("""Answer the question based on the
    context below. If the question cannot be answered using the information
    provided, answer with "I don't know".

Context: {context}

Question: {question}

Answer: """)

model = OllamaLLM(model="llama3.2")

prompt = template.invoke({
    "context": """The most recent advancements in NLP are being driven by Large
        Language Models (LLMs). These models outperform their smaller
        counterparts and have become invaluable for developers who are creating
        applications with NLP capabilities. Developers can tap into these
        models through Hugging Face's `transformers` library, or by utilizing
        OpenAI and Cohere's offerings through the `openai` and `cohere`
        libraries, respectively.""",
    "question": "Which model providers offer LLMs?"
})

completion = model.invoke(prompt)

print(completion)

Hugging Face's `transformers` library, and OpenAI and Cohere's offerings through the `openai` and `cohere` libraries, respectively.


## 코드 1-6 역할에 따른 동적 프롬프트

In [9]:
from langchain_core.prompts import ChatPromptTemplate
template = ChatPromptTemplate.from_messages([
    ('system', '''Answer the question based on the context below. If the
        question cannot be answered using the information provided, answer with
        "I don\'t know".'''),
    ('human', 'Context: {context}'),
    ('human', 'Question: {question}'),
])

template.invoke({
    "context": """The most recent advancements in NLP are being driven by Large
        Language Models (LLMs). These models outperform their smaller
        counterparts and have become invaluable for developers who are creating
        applications with NLP capabilities. Developers can tap into these
        models through Hugging Face's `transformers` library, or by utilizing
        OpenAI and Cohere's offerings through the `openai` and `cohere`
        libraries, respectively.""",
    "question": "Which model providers offer LLMs?"
})

ChatPromptValue(messages=[SystemMessage(content='Answer the question based on the context below. If the\n        question cannot be answered using the information provided, answer with\n        "I don\'t know".', additional_kwargs={}, response_metadata={}), HumanMessage(content="Context: The most recent advancements in NLP are being driven by Large\n        Language Models (LLMs). These models outperform their smaller\n        counterparts and have become invaluable for developers who are creating\n        applications with NLP capabilities. Developers can tap into these\n        models through Hugging Face's `transformers` library, or by utilizing\n        OpenAI and Cohere's offerings through the `openai` and `cohere`\n        libraries, respectively.", additional_kwargs={}, response_metadata={}), HumanMessage(content='Question: Which model providers offer LLMs?', additional_kwargs={}, response_metadata={})])

## 코드 1-7 두 개의 동적 프롬프트를 적용한 호출

In [10]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ('system', '''Answer the question based on the context below. If the
        question cannot be answered using the information provided, answer
        with "I don\'t know".'''),
    ('human', 'Context: {context}'),
    ('human', 'Question: {question}'),
])

model = ChatOllama(model="llama3.2")

prompt = template.invoke({
    "context": """The most recent advancements in NLP are being driven by
        Large Language Models (LLMs). These models outperform their smaller
        counterparts and have become invaluable for developers who are creating
        applications with NLP capabilities. Developers can tap into these
        models through Hugging Face's `transformers` library, or by utilizing
        OpenAI and Cohere's offerings through the `openai` and `cohere`
        libraries, respectively.""",
    "question": "Which model providers offer LLMs?"
})

model.invoke(prompt)

AIMessage(content="According to the context, the model providers that offer Large Language Models (LLMs) are:\n\n1. Hugging Face's `transformers` library\n2. OpenAI\n3. Cohere", additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-03-08T05:27:16.920802542Z', 'done': True, 'done_reason': 'stop', 'total_duration': 26023430351, 'load_duration': 154664204, 'prompt_eval_count': 162, 'prompt_eval_duration': 16331374229, 'eval_count': 41, 'eval_duration': 9517503699, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--019ccbe9-9d0f-7e90-a35f-8b18a433946d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 162, 'output_tokens': 41, 'total_tokens': 203})

## 코드 1-8 JSON 형식 출력 요청

In [11]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel

class AnswerWithJustification(BaseModel):
    '''An answer to the user's question along with justification for the
        answer.'''
    answer: str
    justification: str

llm = ChatOllama(model="llama3.2", temperature=0)
structured_llm = llm.with_structured_output(AnswerWithJustification)

result = structured_llm.invoke("What weighs more, a pound of bricks or a pound of feathers?")

print(result.model_dump_json())


{"answer":"They weigh the same","justification":"One pound is a unit of weight or mass, and it is defined as the weight of a kilogram. Regardless of the density or composition of the object, one pound of any substance will weigh the same amount."}


## 코드 1-9 랭체인의 CSV 출력 파서

In [12]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
parser = CommaSeparatedListOutputParser()
items = parser.invoke("apple, banana, cherry")
print(items)

['apple', 'banana', 'cherry']


## 코드 1-10 랭체인의 공통 인터페이스 예시

In [13]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="llama3.2")

completion = model.invoke('Hi there!')
print(completion)

completions = model.batch(['Hi there!', 'Bye!'])
print(completions)

for token in model.stream('Bye!'):
    print(token, end="| ", flush=True)

content='How can I assist you today?' additional_kwargs={} response_metadata={'model': 'llama3.2', 'created_at': '2026-03-08T05:27:36.9511973Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2953125409, 'load_duration': 188010692, 'prompt_eval_count': 28, 'prompt_eval_duration': 1378986314, 'eval_count': 8, 'eval_duration': 1382135252, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'} id='lc_run--019ccbea-456c-7241-92e9-4439ba11d701-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 28, 'output_tokens': 8, 'total_tokens': 36}
[AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-03-08T05:27:42.160359616Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5205429326, 'load_duration': 344575916, 'prompt_eval_count': 28, 'prompt_eval_duration': 890267504, 'eval_count': 10, 'eval_duration': 1600141713, 'logprobs': None, 'model_name': 'llama3.2', 'mod

## 코드 1-11 명령형 구성 예시

In [14]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain

template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human", "{question}"),
    ]
)

model = ChatOllama(model="llama3.2")

@chain
def chatbot(values):
    prompt = template.invoke(values)
    return model.invoke(prompt)

response = chatbot.invoke({"question": "Which model providers offer LLMs?"})
print(response.content)

Several model providers offer Large Language Models (LLMs). Here are some of the most notable ones:

1. **Hugging Face**: Hugging Face is one of the most popular providers of LLMs. They offer a wide range of pre-trained models, including BERT, RoBERTa, and Longformer. Their model hub is a popular platform for developers to easily integrate LLMs into their applications.
2. **OpenAI**: OpenAI is the company behind the popular LLM model, GPT-3. They also offer a range of smaller LLM models, including GPT-2 and GPT-2 XL.
3. **Google**: Google offers a range of LLM models, including BERT, RoBERTa, and BigBird. Their models are typically used in research and development applications.
4. **Amazon**: Amazon offers a range of LLM models, including the Amazon SageMaker model hub, which provides access to pre-trained models and tools for building and deploying LLMs.
5. **Microsoft**: Microsoft offers a range of LLM models, including the Microsoft Cognitive Toolkit (CNTK) and the Microsoft Azure M

## 코드 1-12 명령형 구성을 사용한 스트리밍 호출 예시

In [15]:
@chain
def chatbot(values):
    prompt = template.invoke(values)
    for token in model.stream(prompt):
        yield token

for part in chatbot.stream({
    "question": "Which model providers offer LLMs?"
}):
    print(part.content, end="", flush=True)


Several model providers offer Large Language Models (LLMs). Here are some of the most notable ones:

1. **Hugging Face**: Hugging Face is a popular platform that offers a wide range of pre-trained LLMs, including popular models like BERT, RoBERTa, and DistilBERT. They also provide a model hub where developers can share and discover new models.
2. **Google AI**: Google AI offers a range of LLMs, including BERT, RoBERTa, and T5. Their models are pre-trained on large datasets and can be fine-tuned for specific NLP tasks.
3. **Microsoft Azure**: Microsoft Azure offers a range of LLMs, including BERT, RoBERTa, and LaMDA. Their models are pre-trained on large datasets and can be deployed in various Azure services.
4. **Amazon SageMaker**: Amazon SageMaker offers a range of LLMs, including BERT, RoBERTa, and T5. Their models are pre-trained on large datasets and can be fine-tuned for specific NLP tasks.
5. **OpenAI**: OpenAI offers a range of LLMs, including the popular GPT-3 model. Their mod

## 코드 1-13 명령형 구성을 사용한 비동기 실행

In [16]:
@chain
async def chatbot(values):
    prompt = await template.ainvoke(values)
    return await model.ainvoke(prompt)

await chatbot.ainvoke({"question": "Which model providers offer LLMs?"})


AIMessage(content='Several model providers offer Large Language Models (LLMs). Here are some of the most notable ones:\n\n1. **Hugging Face**: Hugging Face is a popular platform for LLMs, offering a wide range of models and datasets. Their models are based on popular architectures like BERT, RoBERTa, and DistilBERT.\n2. **Meta AI**: Meta AI offers a range of LLMs, including the popular XLNet and T5 models. Their models are designed for tasks like question answering, text classification, and language translation.\n3. **Google AI**: Google AI offers a range of LLMs, including the popular BERT and T5 models. Their models are designed for tasks like question answering, text classification, and language translation.\n4. **Amazon SageMaker**: Amazon SageMaker offers a range of LLMs, including the popular transformer-based models. Their models are designed for tasks like text classification, sentiment analysis, and language translation.\n5. **Microsoft Azure AI**: Microsoft Azure AI offers a 

## 코드 1-14 선언형 구성 예시

In [17]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human", "{question}"),
    ]
)

model = ChatOllama(model="llama3.2")

chatbot = template | model

response = chatbot.invoke({"question": "Which model providers offer LLMs?"})
print(response.content)

for part in chatbot.stream({"question": "Which model providers offer LLMs?"}):
    print(part.content, end="", flush=True)

Many model providers offer Large Language Models (LLMs). Some of the most notable ones include:

1. **Hugging Face**: Hugging Face is a popular platform that offers a wide range of pre-trained LLMs, including BERT, RoBERTa, and XLNet. They also provide a model hub where developers can upload and share their own models.
2. **Google**: Google offers a range of LLMs, including BERT, RoBERTa, and T5. They also provide the Hugging Face Transformers library, which is a popular open-source library for NLP tasks.
3. **Amazon**: Amazon offers a range of LLMs, including SageMaker's automatic model tuning and the Amazon Comprehend Natural Language service.
4. **Microsoft**: Microsoft offers a range of LLMs, including the Azure Cognitive Services Text Analytics service and the Microsoft Academic Knowledge Graph.
5. **DeepMind**: DeepMind offers a range of LLMs, including the Transformer-XL and the BERT-MLP models.
6. **Meta AI**: Meta AI offers a range of LLMs, including the Meta Model and the BER

## 코드 1-15 선언형 구성을 사용한 스트리밍 호출 예시

In [18]:
chatbot = template | model

for part in chatbot.stream({
    "question": "Which model providers offer LLMs?"
}):
    print(part)


content='Several' additional_kwargs={} response_metadata={} id='lc_run--019ccbf1-1615-7612-b02e-fd05425f4434' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=' model' additional_kwargs={} response_metadata={} id='lc_run--019ccbf1-1615-7612-b02e-fd05425f4434' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=' providers' additional_kwargs={} response_metadata={} id='lc_run--019ccbf1-1615-7612-b02e-fd05425f4434' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=' offer' additional_kwargs={} response_metadata={} id='lc_run--019ccbf1-1615-7612-b02e-fd05425f4434' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=' Large' additional_kwargs={} response_metadata={} id='lc_run--019ccbf1-1615-7612-b02e-fd05425f4434' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=' Language' additional_kwargs={} response_metadata={} id='lc_run--019ccbf1-1615-7612-b02e-fd05425f4434' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
co

## 코드 1-16 선언형 구성을 사용한 비동기 실행

In [19]:
chatbot = template | model

await chatbot.ainvoke({
    "question": "Which model providers offer LLMs?"
})


AIMessage(content="Many model providers offer Large Language Models (LLMs). Here are some of the most popular ones:\n\n1. **Hugging Face**: Hugging Face is a leading provider of LLMs, offering a wide range of models and a user-friendly interface. Their models are pre-trained on various datasets and can be fine-tuned for specific tasks.\n2. **BERT**: BERT (Bidirectional Encoder Representations from Transformers) is an LLM developed by Google. It's a widely used model that has been pre-trained on a large corpus of text data.\n3. **RoBERTa**: RoBERTa (Robustly Optimized BERT Pretraining Approach) is a variant of BERT developed by Facebook AI. It's known for its better performance on certain tasks, such as question answering.\n4. **Longformer**: Longformer is a LLM developed by Hugging Face that uses a novel attention mechanism to handle long-range dependencies in text data.\n5. **T5**: T5 (Text-to-Text Transfer Transformer) is a LLM developed by Google that's designed for transfer learnin